In [ ]:
# IPython magic commands
%load_ext autoreload
%autoreload 2

# Standard library imports
import gc
import json
import os
import sys
from datetime import datetime
from pathlib import Path

# Third-party imports
import cv2
import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.graph_objs as go
import plotly.io as pio
from IPython.display import display
from tqdm.notebook import tqdm

# AEON imports
from aeon.schema.schemas import social02
from swc.aeon.io import api as aeon_api
from swc.aeon.io import video

# Add project path for custom utilities
cwd = os.getcwd()
project_path = os.path.join(cwd, "ProjectAeon", "aeon_scratchpad", "aeon_analysis", "aeon_methods_paper")
sys.path.append(project_path)

# Manual ID tracking validation

Evaluates SLEAP identity tracking quality when the two mice are close together, which RFID cannot
assess: a read from a reader with both mice in range cannot be attributed to either one.

Frames where the mice are close together are sampled at random, displayed zoomed in with their
identity labels, and scored by eye as correct or swapped.

Run one experiment at a time: set the index in the experiment cell, work down the notebook, then
come back and move on to the next one.

# Definitions

In [ ]:
data_dir = Path("/ceph/aeon/aeon/code/scratchpad/methods_paper_data")
save_dir = Path("/ceph/aeon/aeon/code/scratchpad/anaya/id_accuracy_in_proximity_review")
frames_dir = save_dir / "frames"
cache_dir = save_dir / "cache"
figs_dir = save_dir / "figures"
for _d in (save_dir, frames_dir, cache_dir, figs_dir):
    os.makedirs(_d, exist_ok=True)

fps = 50
DISTANCE_THRESH = 50.0   # px, mice closer than this count as a close encounter
MAX_GAP_SEC = 0.5        # s, larger gaps in the position data split a bout
SAMPLE_POOL = 300        # frames rendered per experiment, to review from
TARGET_CERTAIN = 100     # correct or swapped labels to collect before the review stops
MAX_NEST_FRACTION = 0.25  # cap on the share of sampled frames taken inside the nest
NEST_PADDING = 10        # px, padding added around the nest region from the metadata
LOOKUP_FRAMES = 2        # frame periods either side to search for the nearest video frame
CONTEXT_FRAMES = 5       # frames rendered either side of each sampled frame, to step through
CROP_HALF_WIDTH = 100    # px, crop is 2x this, centred between the two mice
DISPLAY_UPSCALE = 4      # integer upscale applied to the crop
MARKER_RADIUS = 9        # px, radius of the identity circles in the upscaled image
JPEG_QUALITY = 92        # clips are 11 images per sample, so store them as JPEG
RANDOM_SEED = 42

# Identities are ordered by animal number, lowest first, which is the tattooed mouse of each pair.
# Colours are BGR for cv2, and are fixed to that ordering so they mean the same thing everywhere.
IDENTITY_LABELS = ["tattooed", "non-tattooed"]
IDENTITY_COLORS = [(0, 200, 0), (0, 140, 255)]  # green, orange

# The subjects in the social period of social0.3-aeon3 have inverted identities in the position
# data, corrected here as in anaya_figures.ipynb
IDENTITY_CORRECTIONS = {
    "social0.3-aeon3": {"BAA-1104516": "BAA-1104519", "BAA-1104519": "BAA-1104516"},
}

In [ ]:
experiments = [
    {"name": "social0.2-aeon3", "presocial_start": '2024-01-31 11:00:00', "presocial_end": '2024-02-08 15:00:00', "social_start": '2024-02-09 16:00:00', "social_end": '2024-02-23 13:00:00', "postsocial_start": '2024-02-25 17:00:00', "postsocial_end": '2024-03-02 14:00:00'},
    {"name": "social0.2-aeon4", "presocial_start": '2024-01-31 11:00:00', "presocial_end": '2024-02-08 15:00:00', "social_start": '2024-02-09 17:00:00', "social_end": '2024-02-23 12:00:00', "postsocial_start": '2024-02-25 18:00:00', "postsocial_end": '2024-03-02 13:00:00'},
    {"name": "social0.3-aeon3", "presocial_start": '2024-06-08 19:00:00', "presocial_end": '2024-06-17 13:00:00', "social_start": '2024-06-25 11:00:00', "social_end": '2024-07-06 13:00:00', "postsocial_start": '2024-07-07 16:00:00', "postsocial_end": '2024-07-14 14:00:00'},
    {"name": "social0.3-aeon4", "presocial_start": '2024-06-08 19:00:00', "presocial_end": '2024-06-17 14:00:00', "social_start": '2024-06-19 12:00:00', "social_end": '2024-07-03 14:00:00', "postsocial_start": '2024-07-04 11:00:00', "postsocial_end": '2024-07-13 12:00:00'},
    {"name": "social0.4-aeon3", "presocial_start": '2024-08-16 17:00:00', "presocial_end": '2024-08-24 10:00:00', "social_start": '2024-08-28 11:00:00', "social_end": '2024-09-09 13:00:00', "postsocial_start": '2024-09-09 18:00:00', "postsocial_end": '2024-09-22 16:00:00'},
    {"name": "social0.4-aeon4", "presocial_start": '2024-08-16 15:00:00', "presocial_end": '2024-08-24 10:00:00', "social_start": '2024-08-28 10:00:00', "social_end": '2024-09-09 01:00:00', "postsocial_start": '2024-09-09 15:00:00', "postsocial_end": '2024-09-22 16:00:00'}
]

experiment = experiments[0]
print(experiment["name"])

## Load data

Frames are sampled from the whole arena, nest included.

In [ ]:
def video_root(
    experiment: dict
) -> str:
    """
    Get the raw data root path for an experiment.

    Parameters:
    - experiment: Experiment dict with a 'name' key in format 'experiment-computer'

    Returns:
    - Path string, e.g. '/ceph/aeon/aeon/data/raw/AEON3/social0.2'
    """

    exp, acquisition_computer = experiment["name"].split("-", 1)
    return f"/ceph/aeon/aeon/data/raw/{acquisition_computer.upper()}/{exp}"


def sort_identities(
    names: list[str]
) -> list[str]:
    """
    Sort identity names by animal number, lowest first.

    The lowest numbered mouse of each pair is the tattooed one, so this ordering is what
    IDENTITY_LABELS and IDENTITY_COLORS are keyed to.

    Parameters:
    - names: Identity names, e.g. ['BAA-1104047', 'BAA-1104045']

    Returns:
    - The names sorted by their trailing number
    """

    def number(name):
        digits = "".join(c for c in str(name).split("-")[-1] if c.isdigit())
        return (int(digits) if digits else 0, str(name))

    return sorted(names, key=number)


def load_position_pair(
    experiment: dict,
    data_dir: Path,
    period: str = "social",
    cache_dir: Path | None = None
) -> tuple[pd.DataFrame, list[str]]:
    """
    Load denoised position data and reshape to one row per timestamp.

    Parameters:
    - experiment: Experiment dict with a 'name' key
    - data_dir: Directory containing the *_positiondenoised.parquet files
    - period: Experiment period to load (default: 'social')
    - cache_dir: Directory to cache the reshaped table in, skipping the merge on re-runs

    Returns:
    - Tuple of (wide DataFrame with columns ['time', 'x0', 'y0', 'x1', 'y1'],
      sorted list of the two identity names mapping to the 0/1 column suffixes)
    """

    cache_path = ids_path = None
    if cache_dir is not None:
        cache_path = Path(cache_dir) / f"{experiment['name']}_{period}_wide.parquet"
        ids_path = cache_path.with_suffix(".identities.json")
        if cache_path.exists() and ids_path.exists():
            wide = pd.read_parquet(cache_path)
            identities = json.loads(ids_path.read_text())
            print(f"Loaded cache: {len(wide):,} timestamps, identities {identities}")
            return wide, identities

    # Read only the columns needed; the full table is ~9 GB in memory for a 14 day experiment
    pattern = f"{experiment['name']}_{period}_positiondenoised.parquet"
    paths = sorted(Path(data_dir).glob(pattern))
    if not paths:
        raise FileNotFoundError(f"No files matching {pattern} in {data_dir}")

    df = pd.concat(
        [pd.read_parquet(p, columns=["time", "identity_name", "x", "y"]) for p in paths],
        ignore_index=True,
    )
    correction = IDENTITY_CORRECTIONS.get(experiment["name"]) if period == "social" else None
    if correction:
        df["identity_name"] = df["identity_name"].astype(str).replace(correction)
        print(f"Applied identity correction for {experiment['name']}")

    df["identity_name"] = df["identity_name"].astype("category")
    identities = sort_identities(map(str, df["identity_name"].cat.categories))
    if len(identities) != 2:
        raise ValueError(f"Expected exactly two identities, found: {identities}")
    print(f"{len(df):,} rows, identities {identities}")

    # Split by identity and merge on time, so both mice sit on one row
    parts = []
    for i, ident in enumerate(identities):
        part = (
            df.loc[df["identity_name"] == ident, ["time", "x", "y"]]
            .rename(columns={"x": f"x{i}", "y": f"y{i}"})
            .drop_duplicates(subset="time")
        )
        parts.append(part)
    del df
    gc.collect()

    wide = (
        pd.merge(parts[0], parts[1], on="time", how="outer")
        .sort_values("time")
        .reset_index(drop=True)
    )
    del parts
    gc.collect()
    print(f"{len(wide):,} unique timestamps")

    if cache_path is not None:
        wide.to_parquet(cache_path, compression="snappy", index=False)
        ids_path.write_text(json.dumps(identities))

    return wide, identities


wide, identities = load_position_pair(experiment, data_dir, period="social", cache_dir=cache_dir)
for label, ident in zip(IDENTITY_LABELS, identities):
    print(f"{label}: {ident}")
wide.head()

### 1. Find close-proximity bouts

In [ ]:
def find_proximity_bouts(
    wide: pd.DataFrame,
    distance_threshold: float = DISTANCE_THRESH,
    max_gap_s: float = MAX_GAP_SEC
) -> pd.DataFrame:
    """
    Find contiguous periods where the two mice are closer than a threshold.

    Parameters:
    - wide: DataFrame with columns ['time', 'x0', 'y0', 'x1', 'y1']
    - distance_threshold: Maximum distance in pixels for considering mice "close"
    - max_gap_s: Gaps larger than this in the position data split a bout in two, so a bout
      is a continuous period rather than one spanning an acquisition gap

    Returns:
    - DataFrame with columns ['start_idx', 'end_idx', 'start_time', 'end_time',
      'n_frames', 'duration_s'], where end_idx is inclusive
    """

    times = wide["time"].to_numpy()
    d = np.hypot(
        wide["x0"].to_numpy() - wide["x1"].to_numpy(),
        wide["y0"].to_numpy() - wide["y1"].to_numpy(),
    )
    close = np.isfinite(d) & (d < distance_threshold)

    # A bout starts where a close frame follows a non-close frame or a data gap, and vice versa
    dt = np.diff(times) / np.timedelta64(1, "s")
    gap_before = np.r_[True, dt > max_gap_s]
    gap_after = np.r_[dt > max_gap_s, True]
    prev_close = np.r_[False, close[:-1]]
    next_close = np.r_[close[1:], False]

    starts = np.flatnonzero(close & (~prev_close | gap_before))
    ends = np.flatnonzero(close & (~next_close | gap_after))

    bouts = pd.DataFrame({
        "start_idx": starts,
        "end_idx": ends,
        "start_time": times[starts],
        "end_time": times[ends],
    })
    bouts["n_frames"] = bouts["end_idx"] - bouts["start_idx"] + 1
    bouts["duration_s"] = (bouts["end_time"] - bouts["start_time"]) / np.timedelta64(1, "s")
    return bouts


bouts = find_proximity_bouts(wide)
pct_close = bouts["n_frames"].sum() / len(wide) * 100
print(f"{len(bouts):,} proximity bouts (<{DISTANCE_THRESH:.0f} px)")
print(f"{pct_close:.1f}% of all timestamps are close encounters")
print(f"Median bout duration: {bouts['duration_s'].median():.2f} s, "
      f"longest: {bouts['duration_s'].max():.1f} s")

### 2. Sample one frame per bout

Sampling at the bout level rather than the frame level, so a single long huddle does not dominate
the sample. More frames are sampled than will be reviewed, since frames scored unsure do not count
towards the target.

Most close encounters happen in the nest, so nest bouts are capped at `MAX_NEST_FRACTION` of the
sample to keep the rest of the arena represented.

In [ ]:
def nest_bounds(
    experiment: dict,
    padding: float = NEST_PADDING
) -> tuple[float, float, float, float]:
    """
    Get the nest bounding box from the experiment metadata.

    Parameters:
    - experiment: Experiment dict with a 'name' key
    - padding: Pixels to expand the box by on each side

    Returns:
    - Tuple of (x_min, x_max, y_min, y_max) in full-frame pixels
    """

    metadata = aeon_api.load(video_root(experiment), social02.Metadata)["metadata"].iloc[0]
    points = [(float(p.X), float(p.Y))
              for p in metadata.ActiveRegion.NestRegion.ArrayOfPoint]
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    return min(xs) - padding, max(xs) + padding, min(ys) - padding, max(ys) + padding


def flag_nest_bouts(
    bouts: pd.DataFrame,
    wide: pd.DataFrame,
    bounds: tuple[float, float, float, float]
) -> np.ndarray:
    """
    Flag bouts where the mice are in the nest, judged at the midpoint of each bout.

    Parameters:
    - bouts: DataFrame of proximity bouts from find_proximity_bouts
    - wide: DataFrame with columns ['time', 'x0', 'y0', 'x1', 'y1']
    - bounds: Nest bounding box as (x_min, x_max, y_min, y_max)

    Returns:
    - Boolean array, one entry per bout
    """

    x_min, x_max, y_min, y_max = bounds
    mid = ((bouts["start_idx"].to_numpy() + bouts["end_idx"].to_numpy()) // 2)
    mid_rows = wide.iloc[mid]
    x = np.nanmean([mid_rows["x0"].to_numpy(), mid_rows["x1"].to_numpy()], axis=0)
    y = np.nanmean([mid_rows["y0"].to_numpy(), mid_rows["y1"].to_numpy()], axis=0)
    return (x >= x_min) & (x <= x_max) & (y >= y_min) & (y <= y_max)


def sample_frames_from_bouts(
    bouts: pd.DataFrame,
    wide: pd.DataFrame,
    n: int = SAMPLE_POOL,
    seed: int = RANDOM_SEED,
    is_nest: np.ndarray | None = None,
    max_nest_fraction: float = MAX_NEST_FRACTION
) -> pd.DataFrame:
    """
    Sample n bouts at random and take one random frame from each.

    Parameters:
    - bouts: DataFrame of proximity bouts from find_proximity_bouts
    - wide: DataFrame with columns ['time', 'x0', 'y0', 'x1', 'y1']
    - n: Number of frames to sample
    - seed: Seed for the random generator, so the sample is reproducible
    - is_nest: Boolean array flagging nest bouts, from flag_nest_bouts; None to sample
      without any cap on nest frames
    - max_nest_fraction: Maximum share of the sample taken from nest bouts

    Returns:
    - DataFrame with columns ['frame_time', 'x0', 'y0', 'x1', 'y1', 'bout_start',
      'bout_end', 'duration_s', 'in_nest']
    """

    rng = np.random.default_rng(seed)
    n_bouts = len(bouts)
    if n_bouts == 0:
        raise ValueError("No proximity bouts found")

    if is_nest is None:
        is_nest = np.zeros(n_bouts, dtype=bool)

    # Fill the non-nest quota first, then top up with nest bouts to the cap
    nest_pool = np.flatnonzero(is_nest)
    other_pool = np.flatnonzero(~is_nest)
    n_other = min(len(other_pool), n - int(round(n * max_nest_fraction)))
    n_nest = min(len(nest_pool), n - n_other)
    n_other = min(len(other_pool), n - n_nest)

    chosen = np.concatenate([
        rng.choice(other_pool, size=n_other, replace=False),
        rng.choice(nest_pool, size=n_nest, replace=False),
    ])
    if len(chosen) < n:
        print(f"Only {len(chosen)} bouts available for {n} samples")

    rows = []
    for bout_i in sorted(chosen):
        b = bouts.iloc[int(bout_i)]
        span = np.arange(int(b.start_idx), int(b.end_idx) + 1)
        idx = int(rng.choice(span))
        w = wide.iloc[idx]
        rows.append({
            "frame_time": w["time"],
            "x0": w["x0"], "y0": w["y0"],
            "x1": w["x1"], "y1": w["y1"],
            "bout_start": b.start_time,
            "bout_end": b.end_time,
            "duration_s": b.duration_s,
            "in_nest": bool(is_nest[int(bout_i)]),
        })

    return pd.DataFrame(rows).sort_values("frame_time").reset_index(drop=True)


bounds = nest_bounds(experiment)
is_nest = flag_nest_bouts(bouts, wide, bounds)
print(f"Nest region: x {bounds[0]:.0f}-{bounds[1]:.0f}, y {bounds[2]:.0f}-{bounds[3]:.0f}")
print(f"{is_nest.mean():.1%} of all bouts are in the nest")

sample_df = sample_frames_from_bouts(bouts, wide, n=SAMPLE_POOL, seed=RANDOM_SEED, is_nest=is_nest)
print(f"Sampled {len(sample_df)} frames, {sample_df['in_nest'].mean():.1%} in the nest, spanning "
      f"{sample_df['frame_time'].min()} to {sample_df['frame_time'].max()}")
sample_df.head()

### 3. Render cropped clips

A short clip is rendered around each sampled frame, `CONTEXT_FRAMES` either side, so the mice can be
followed through the encounter in the GUI. The crop box is fixed on the middle frame so the view does
not move while stepping.

Rendering up front rather than inside the GUI, since seeking the raw video is slow. Existing files
are skipped, so this cell is safe to re-run.

In [ ]:
def render_review_clip(
    root: str,
    frame_time: pd.Timestamp,
    wide: pd.DataFrame,
    half_width: int = CROP_HALF_WIDTH,
    upscale: int = DISPLAY_UPSCALE,
    context_frames: int = CONTEXT_FRAMES,
    lookup_frames: int = LOOKUP_FRAMES
) -> tuple[list[np.ndarray], int] | None:
    """
    Render a short clip around a timestamp, cropped on the mice and marked with identity circles.

    Parameters:
    - root: Root directory of the raw data
    - frame_time: Timestamp at the middle of the clip
    - wide: DataFrame with columns ['time', 'x0', 'y0', 'x1', 'y1']
    - half_width: Half the side length of the square crop, in full-frame pixels
    - upscale: Integer upscale factor applied to the crop
    - context_frames: Frames rendered either side of frame_time
    - lookup_frames: Extra frame periods loaded, to allow for timestamps that do not land
      exactly on a camera frame

    Returns:
    - Tuple of (list of BGR images, index of the middle frame), or None if no video frame
      is found around the timestamp
    """

    # One load covering the whole clip, so the video is seeked once and decoded sequentially
    span = pd.Timedelta(seconds=(context_frames + lookup_frames + 1) / fps)
    info = aeon_api.load(root, social02.CameraTop.Video,
                         start=frame_time - span, end=frame_time + span)
    if len(info) == 0:
        return None

    centre = int(np.argmin(np.abs(info.index - frame_time)))
    lo = max(0, centre - context_frames)
    hi = min(len(info), centre + context_frames + 1)
    selection = info.iloc[lo:hi]
    centre_index = centre - lo

    # Positions for every frame in the clip, so the circles track the mice as they move
    times = pd.to_datetime(selection.index)
    margin = pd.Timedelta(seconds=1.0)
    pos_times = pd.Series(pd.to_datetime(wide["time"]))
    slice_lo = pos_times.searchsorted(times[0] - margin)
    slice_hi = pos_times.searchsorted(times[-1] + margin)
    matched = pd.merge_asof(
        pd.DataFrame({"time": times}),
        wide.iloc[slice_lo:slice_hi][["time", "x0", "y0", "x1", "y1"]],
        on="time", direction="nearest", tolerance=pd.Timedelta(seconds=1.0 / (2 * fps)),
    )

    # Fix the crop on the middle frame so the view does not move while stepping through
    middle = matched.iloc[centre_index]
    valid = [(middle[f"x{i}"], middle[f"y{i}"]) for i in range(2)
             if np.isfinite(middle[f"x{i}"]) and np.isfinite(middle[f"y{i}"])]
    if not valid:
        return None
    centre_x = int(np.mean([p[0] for p in valid]))
    centre_y = int(np.mean([p[1] for p in valid]))

    images = []
    side = 2 * half_width
    x0 = y0 = None
    for k, frame in enumerate(video.frames(selection)):
        if frame.ndim == 2:
            frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
        if x0 is None:
            height, width = frame.shape[:2]
            x0 = int(np.clip(centre_x - half_width, 0, max(0, width - side)))
            y0 = int(np.clip(centre_y - half_width, 0, max(0, height - side)))

        crop = frame[y0:y0 + side, x0:x0 + side].copy()
        if upscale > 1:
            crop = cv2.resize(crop, None, fx=upscale, fy=upscale, interpolation=cv2.INTER_LINEAR)

        row = matched.iloc[k]
        for i in range(2):
            x, y = row[f"x{i}"], row[f"y{i}"]
            if not (np.isfinite(x) and np.isfinite(y)):
                continue
            px = (int(x) - x0) * upscale
            py = (int(y) - y0) * upscale
            if not (0 <= px < crop.shape[1] and 0 <= py < crop.shape[0]):
                continue
            # Dark ring under the colour, so the marker reads on the light arena and dark nest
            cv2.circle(crop, (px, py), MARKER_RADIUS, (0, 0, 0), 4)
            cv2.circle(crop, (px, py), MARKER_RADIUS, IDENTITY_COLORS[i], 2)
        images.append(crop)

    return images, centre_index


def prerender_review_clips(
    sample_df: pd.DataFrame,
    experiment: dict,
    wide: pd.DataFrame,
    identities: list[str],
    frames_dir: Path
) -> pd.DataFrame:
    """
    Render a clip per sampled frame and return a manifest for the review GUI.

    Parameters:
    - sample_df: DataFrame of sampled frames from sample_frames_from_bouts
    - experiment: Experiment dict with a 'name' key
    - wide: DataFrame with columns ['time', 'x0', 'y0', 'x1', 'y1']
    - identities: Identity names ordered by sort_identities, tattooed mouse first
    - frames_dir: Directory to save images to; a per-experiment subfolder is created

    Returns:
    - DataFrame with columns ['experiment_name', 'frame_time', 'image_path', 'clip_paths',
      'centre_index', 'in_nest', 'tattooed', 'non_tattooed']
    """

    root = video_root(experiment)
    out_dir = Path(frames_dir) / experiment["name"]
    os.makedirs(out_dir, exist_ok=True)

    records, n_missing = [], 0
    for row in tqdm(list(sample_df.itertuples()), desc="Rendering clips"):
        frame_time = pd.Timestamp(row.frame_time)
        stem = f"{frame_time:%Y%m%d_%H%M%S_%f}"
        existing = sorted(out_dir.glob(f"{stem}_*.jpg"))

        if existing:
            clip_paths = [str(p) for p in existing]
            centre_index = len(clip_paths) // 2
        else:
            rendered = render_review_clip(root, frame_time, wide)
            if rendered is None:
                n_missing += 1
                continue
            images, centre_index = rendered
            clip_paths = []
            for k, img in enumerate(images):
                path = out_dir / f"{stem}_{k:02d}.jpg"
                cv2.imwrite(str(path), img, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
                clip_paths.append(str(path))

        records.append({
            "experiment_name": experiment["name"],
            "frame_time": frame_time,
            "image_path": clip_paths[centre_index],
            "clip_paths": clip_paths,
            "centre_index": centre_index,
            "in_nest": row.in_nest,
            "tattooed": identities[0],
            "non_tattooed": identities[1],
        })

    if n_missing:
        print(f"Skipped {n_missing} frames with no video frame in the search window")
    return pd.DataFrame(records)


manifest = prerender_review_clips(sample_df, experiment, wide, identities, frames_dir)
print(f"{len(manifest)} clips ready in {frames_dir / experiment['name']}")
manifest.head()

### 4. Manual review GUI

Step through the clip with the slider or the arrow buttons to watch the mice move, which usually
makes it obvious which circle is following which animal. Then click **Correct** if each circle is on
the right mouse, **Swapped** if they are on the wrong animals, or **Unsure** if the clip is too
occluded to call. **Back** returns to the previous clip.

The review stops once `TARGET_CERTAIN` clips have been scored correct or swapped; unsure clips do not
count towards the target. Labels are written to CSV on every click, so leaving and coming back
resumes at the first unlabelled clip.

In [ ]:
class ReviewGUI:
    """
    Click-through review of pre-rendered frames, saving labels on every click.

    Parameters:
    - manifest: DataFrame of rendered clips from prerender_review_clips
    - csv_path: Path to save labels to, read back on construction to resume a part-finished review
    - identities: Sorted identity names, for the colour key
    - target_certain: Number of correct or swapped labels to collect before stopping
    """

    LABELS = ["correct", "swapped", "unsure"]
    STYLES = ["success", "danger", "warning"]

    def __init__(
        self,
        manifest: pd.DataFrame,
        csv_path: Path,
        identities: list[str],
        target_certain: int = TARGET_CERTAIN
    ):
        self.manifest = manifest.reset_index(drop=True)
        self.csv_path = Path(csv_path)
        self.identities = identities
        self.target_certain = target_certain
        self.labels = self._load_existing()
        self.i = self._first_unlabelled()

        self.header = widgets.HTML()
        self.image = widgets.Image(format="jpeg")
        self.status = widgets.HTML()
        self.key = widgets.HTML(self._colour_key())

        # Clip stepping
        self.slider = widgets.IntSlider(min=0, max=1, value=0, readout=False,
                                        layout=widgets.Layout(width="260px"))
        self.slider.observe(self._on_slide, names="value")
        prev_frame = widgets.Button(description="\u25c0", layout=widgets.Layout(width="45px"))
        next_frame = widgets.Button(description="\u25b6", layout=widgets.Layout(width="45px"))
        prev_frame.on_click(lambda _: self._step(-1))
        next_frame.on_click(lambda _: self._step(1))
        self.frame_label = widgets.HTML()
        self.clip_controls = widgets.HBox([prev_frame, next_frame, self.slider, self.frame_label])

        buttons = []
        for label, style in zip(self.LABELS, self.STYLES):
            button = widgets.Button(description=label.capitalize(), button_style=style,
                                    layout=widgets.Layout(width="130px"))
            button.on_click(self._make_handler(label))
            buttons.append(button)
        back = widgets.Button(description="\u2190 Back", layout=widgets.Layout(width="100px"))
        back.on_click(self._on_back)

        self.controls = widgets.VBox([self.clip_controls, widgets.HBox(buttons + [back])])
        self.widget = widgets.VBox([self.header, self.key, self.image, self.controls, self.status])
        self._load_clip()

    def _colour_key(
        self
    ) -> str:
        """Build the colour key shown above the image."""

        parts = []
        for i, label in enumerate(IDENTITY_LABELS):
            b, g, r = IDENTITY_COLORS[i % len(IDENTITY_COLORS)]
            parts.append(
                f'<span style="color:rgb({r},{g},{b});font-size:20px">&#9679;</span> '
                f'<span style="font-size:14px">{label}</span>'
            )
        return "&nbsp;&nbsp;&nbsp;&nbsp;".join(parts)

    def _clip_paths(
        self,
        i: int
    ) -> list[str]:
        """Get the clip image paths for a manifest row."""

        paths = self.manifest.iloc[i]["clip_paths"]
        return list(paths) if isinstance(paths, (list, tuple)) else json.loads(paths)

    def _load_clip(
        self
    ) -> None:
        """Point the slider at the current clip and show its middle frame."""

        if self.i < len(self.manifest):
            paths = self._clip_paths(self.i)
            self.slider.max = len(paths) - 1
            centre = int(self.manifest.iloc[self.i]["centre_index"])
            # Setting the value fires _on_slide, which draws the frame
            if self.slider.value == centre:
                self._show_frame()
            else:
                self.slider.value = centre
        self._refresh()

    def _step(
        self,
        delta: int
    ) -> None:
        """Move the slider by delta frames, staying in range."""

        self.slider.value = int(np.clip(self.slider.value + delta, self.slider.min, self.slider.max))

    def _on_slide(
        self,
        change
    ) -> None:
        """Show a different frame of the current clip."""

        self._show_frame()

    def _show_frame(
        self
    ) -> None:
        """Load the selected frame of the current clip into the image widget."""

        if self.i >= len(self.manifest):
            return
        paths = self._clip_paths(self.i)
        k = int(np.clip(self.slider.value, 0, len(paths) - 1))
        self.image.value = Path(paths[k]).read_bytes()
        centre = int(self.manifest.iloc[self.i]["centre_index"])
        self.frame_label.value = f"&nbsp;frame {k - centre:+d}" if k != centre else "&nbsp;middle"

    def _key(
        self,
        i: int
    ) -> str:
        """Get the label dict key for a manifest row."""

        return str(pd.Timestamp(self.manifest.iloc[i]["frame_time"]))

    def _load_existing(
        self
    ) -> dict:
        """Load labels from a previous session, keyed by frame time."""

        if not self.csv_path.exists():
            return {}
        prev = pd.read_csv(self.csv_path)
        labels = {str(pd.Timestamp(t)): l for t, l in zip(prev["frame_time"], prev["label"])}
        print(f"Resuming: {len(labels)} frames already labelled")
        return labels

    def _first_unlabelled(
        self
    ) -> int:
        """Get the index of the first frame without a label."""

        for i in range(len(self.manifest)):
            if self._key(i) not in self.labels:
                return i
        return len(self.manifest)

    def _n_certain(
        self
    ) -> int:
        """Count labels that are not unsure."""

        return sum(1 for v in self.labels.values() if v in ("correct", "swapped"))

    def _save(
        self
    ) -> None:
        """Write all labels collected so far to CSV."""

        out = self.manifest.drop(columns=["clip_paths"])
        out["label"] = [self.labels.get(self._key(i)) for i in range(len(out))]
        out["labelled_at"] = datetime.now().isoformat(timespec="seconds")
        out.dropna(subset=["label"]).to_csv(self.csv_path, index=False)

    def _make_handler(
        self,
        label: str
    ):
        """Build the click handler that records a label and advances."""

        def handler(_):
            if self.i >= len(self.manifest):
                return
            self.labels[self._key(self.i)] = label
            self._save()
            self.i += 1
            self._load_clip()

        return handler

    def _on_back(
        self,
        _
    ) -> None:
        """Step back to the previous clip."""

        self.i = max(0, self.i - 1)
        self._load_clip()

    def _refresh(
        self
    ) -> None:
        """Update the header and controls, or show the final summary."""

        n_certain = self._n_certain()
        done = len(self.labels)

        if n_certain >= self.target_certain or self.i >= len(self.manifest):
            self.controls.layout.display = "none"
            self.image.layout.display = "none"
            self.key.layout.display = "none"
            counts = pd.Series(list(self.labels.values())).value_counts().to_dict()
            summary = " \u00b7 ".join(f"{k}: {v}" for k, v in counts.items())
            reached = "target reached" if n_certain >= self.target_certain else "pool exhausted"
            self.header.value = (f"<b>Review complete, {reached}</b>: "
                                 f"{n_certain}/{self.target_certain} certain, {done} labelled")
            self.status.value = f"<code>{summary}</code><br>Saved to {self.csv_path}"
            return

        self.controls.layout.display = ""
        row = self.manifest.iloc[self.i]
        existing = self.labels.get(self._key(self.i))
        current = f" \u2014 currently <b>{existing}</b>" if existing else ""
        where = "nest" if row.get("in_nest") else "arena"
        self.header.value = (
            f"<b>{n_certain} / {self.target_certain} certain</b> \u00b7 "
            f"clip {self.i + 1} of {len(self.manifest)} \u00b7 {where} \u00b7 "
            f"{pd.Timestamp(row['frame_time'])}{current}"
        )
        self.status.value = f"{done} labelled, {done - n_certain} unsure"


review_csv = save_dir / f"id_review_{experiment['name']}.csv"
gui = ReviewGUI(manifest, review_csv, identities)
display(gui.widget)

### 5. Results

Reads every review CSV on disk, so this works without re-running the cells above. Accuracy is
correct / (correct + swapped); unsure frames are excluded from the denominator and reported
separately.

In [ ]:
review_files = sorted(save_dir.glob("id_review_*.csv"))
if not review_files:
    raise FileNotFoundError(f"No review CSVs in {save_dir}")
reviews = pd.concat([pd.read_csv(f) for f in review_files], ignore_index=True)

# Per-experiment counts and accuracy
summary = (
    reviews.groupby("experiment_name")["label"].value_counts().unstack(fill_value=0)
    .reindex(columns=ReviewGUI.LABELS, fill_value=0)
)
summary["n_reviewed"] = summary[ReviewGUI.LABELS].sum(axis=1)
summary["n_scored"] = summary["correct"] + summary["swapped"]
summary["id_accuracy"] = summary["correct"] / summary["n_scored"].replace(0, np.nan)
summary["unsure_rate"] = summary["unsure"] / summary["n_reviewed"]
display(summary)

accuracy = summary["id_accuracy"].dropna()
mean_accuracy = accuracy.mean()
sem_accuracy = accuracy.std(ddof=1) / np.sqrt(len(accuracy)) if len(accuracy) > 1 else np.nan
print(f"Mean ID accuracy: {mean_accuracy:.2%} (SEM: {sem_accuracy:.2%}, n = {len(accuracy)})")
print(f"Mean unsure rate: {summary['unsure_rate'].mean():.2%}")

fig = go.Figure()
fig.add_trace(go.Bar(
    x=summary.index,
    y=summary["id_accuracy"],
    text=[f"{v:.1%}" for v in summary["id_accuracy"]],
    textposition="outside",
    marker=dict(color="#4C78A8"),
))
fig.update_layout(
    height=450,
    width=800,
    title="ID accuracy on close-proximity frames",
    xaxis_title="Experiment",
    yaxis_title="ID accuracy",
    yaxis=dict(range=[0, 1.05], tickformat=".0%"),
    showlegend=False,
)
fig.show()

# svg_path = figs_dir / "manual_id_accuracy_per_experiment.svg"
# pio.write_image(fig, str(svg_path), format="svg")

In [ ]:
# Mean across experiments
fig = go.Figure()
fig.add_trace(go.Bar(
    x=["Mean ID accuracy"],
    y=[mean_accuracy],
    error_y=dict(type="data", array=[sem_accuracy]),
    marker=dict(color="#4C78A8"),
))
fig.update_layout(
    showlegend=False,
    width=300,
    height=400,
    yaxis=dict(range=[0, 1.05], tickformat=".0%"),
    margin=dict(t=20, b=20),
)
fig.update_xaxes(title_text="", showticklabels=True)
fig.update_yaxes(title_text="")
fig.show()

# svg_path = figs_dir / "manual_id_accuracy_mean.svg"
# pio.write_image(fig, str(svg_path), format="svg")

# Save results
results_path = save_dir / "manual_id_review_results.json"
records = [
    {
        "name": name,
        "n_reviewed": int(row["n_reviewed"]),
        "n_correct": int(row["correct"]),
        "n_swapped": int(row["swapped"]),
        "n_unsure": int(row["unsure"]),
        "id_accuracy": None if pd.isna(row["id_accuracy"]) else float(row["id_accuracy"]),
    }
    for name, row in summary.iterrows()
]
with open(results_path, "w") as f:
    json.dump(records, f, indent=2)
print(f"Wrote {results_path}")